In [ ]:
#| default_exp network

In [ ]:
#| export

from __future__ import annotations

from collections import defaultdict
from dataclasses import replace
from typing import Dict, List

from comm_core.backend import CommunicationBackend
from comm_core.core import Message, Node, TransmissionResult
from comm_core.metrics import MetricsCollector
from comm_core.protocol import Protocol
from comm_core.topology import Topology

In [ ]:
#| export

class Network:

    def __init__(
        self,
        nodes: List[Node],
        topology: Topology,
        protocol: Protocol,
        backend: CommunicationBackend,
        metrics: MetricsCollector | None = None,
    ):
        self.nodes = {
            node.id: node
            for node in nodes
        }

        self.topology = topology
        self.protocol = protocol
        self.backend = backend

        self.metrics = (
            metrics
            if metrics is not None
            else MetricsCollector()
        )

        self._inbox: Dict[str, List[Message]] = (
            defaultdict(list)
        )

        self.time = 0.0

    def reset(self, seed: int | None = None) -> None:
        "Start over (e.g. a new episode): clock, queues, inboxes and metrics; `seed` reseeds the backend."
        self.time = 0.0
        self._inbox.clear()
        self.protocol.reset()
        self.backend.reset(seed)
        self.metrics.reset()

    def send(
        self,
        message: Message,
    ) -> None:

        if message.sender not in self.nodes:
            raise ValueError(
                f"Unknown sender: {message.sender}"
            )

        if message.receiver not in self.nodes:
            raise ValueError(
                f"Unknown receiver: {message.receiver}"
            )

        if not self.topology.is_connected(
            message.sender,
            message.receiver,
        ):
            raise ValueError(
                f"No link from "
                f"{message.sender} to "
                f"{message.receiver}"
            )

        self.protocol.submit(
            message,
            self.time,
        )

    def step(
        self,
        dt: float = 1.0,
    ) -> List[TransmissionResult]:

        self.time += dt

        self.backend.step(self.time)

        transmissions = self.protocol.schedule(
            self.time
        )

        results = []

        for transmission in transmissions:

            link = self.topology.get_link(
                transmission.sender,
                transmission.receiver,
            )

            result = self.backend.transmit(
                transmission,
                link,
            )

            self.metrics.record(
                transmission,
                result,
            )

            if result.success:

                # what the receiver gets is what the channel delivered (it may differ from what was sent)
                self._inbox[
                    transmission.receiver
                ].append(
                    replace(transmission.message, payload=result.payload)
                )

            results.append(result)

        return results

    def receive(
        self,
        node_id: str,
    ) -> List[Message]:

        messages = self._inbox[node_id]

        self._inbox[node_id] = []

        return messages

    

### Tests

In [ ]:
from comm_core.channel import AWGNChannel, Channel, IdentityChannel
from comm_core.backend import AnalyticalBackend
from comm_core.core import Message, TransmissionResult
from comm_core.protocol import IdealProtocol
from comm_core.topology import FullMeshTopology, StarTopology

nodes = [Node(n) for n in ('a', 'b', 'c')]
net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(IdentityChannel()))
net.send(Message('a', 'b', payload=[1, 2], size_bits=64))
net.send(Message('c', 'b', payload=[3], size_bits=32))
results = net.step()
assert [r.success for r in results] == [True, True] and net.receive('b')[1].payload == [3]
assert net.receive('b') == [] and net.metrics.summary().bits_received == 96
# the receiver gets what the channel delivered
class Flip(Channel):
    def transmit(self, transmission, link):
        return TransmissionResult(success=True, transmission=transmission, payload=-transmission.message.payload)
net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(Flip()))
net.send(Message('a', 'c', payload=5))
net.step()
assert net.receive('c')[0].payload == -5
# reset: a new episode, replayed with the same seed
def episode(net, seed):
    net.reset(seed)
    out = []
    for _ in range(20):
        net.send(Message('a', 'b', payload=0))
        out += [r.success for r in net.step()]
    return out
net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(AWGNChannel(5, 0.5)))
first = episode(net, 0)
assert episode(net, 0) == first and net.time == 20.0 and net.metrics.summary().transmissions == 20
# messages need a link
net = Network(nodes, StarTopology('a', ['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(IdentityChannel()))
try:
    net.send(Message('b', 'c', payload=0)); raise AssertionError('no error')
except ValueError as e:
    assert 'No link' in str(e)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()